In [151]:
#/root/jupyter_notebooks/mvm-accelerator/test.ipynb

import numpy as np
from pynq import Overlay, allocate, MMIO, PL
import time, re, os, sys

### Config

In [152]:
VERBOSE = 3 # <0, 1, 2, 3, 4>
PROFILE = 1 # <0, 1>

ELEMENT_WIDTH = 16 # <16, 32, 64>
MATRIX_DIM    = 512

In [153]:
BASE_DIR = "/home/ubuntu/mvm-accelerator"
DATA_DIR = "Data" if MATRIX_DIM == 17088 else "Data_Country" if MATRIX_DIM == 192 else "mat-gen"

SCALE     = "scaled" # <reduced, scaled> ~ only matters if DATA_DIR != "mat-gen"
FILE_TYPE = "npy"    # <bin, npy>

In [154]:
# the following params are fixed in hardware

CDMA_BASE = 0xA0000000 
BRAM_BASE = 0x80000000 
PROF_BASE = 0xA00C0000

AXIS_CLK_HZ = 250e6
AXIS_CLK_MHZ = AXIS_CLK_HZ / 1e6

AVAILABLE_CHANNELS = 4
ACTIVE_CHANNELS = AVAILABLE_CHANNELS
NUM_PARTITIONS  = AVAILABLE_CHANNELS

NUM_ROWS         = MATRIX_DIM
ROWS_PER_CHANNEL = NUM_ROWS // ACTIVE_CHANNELS

WORD_WIDTH = 128
ELEMENTS_PER_WORD = WORD_WIDTH // ELEMENT_WIDTH

BYTES_PER_ELEMENT = ELEMENT_WIDTH // 8
BYTES_PER_WORD = WORD_WIDTH // 8

ELEMENTS_PER_ROW = MATRIX_DIM
WORDS_PER_ROW = ELEMENTS_PER_ROW // ELEMENTS_PER_WORD
BYTES_PER_ROW = WORDS_PER_ROW * BYTES_PER_WORD

ELEMENTS_PER_PARTITION = ELEMENTS_PER_ROW // NUM_PARTITIONS
BYTES_PER_PARTITION = BYTES_PER_ROW // NUM_PARTITIONS
PARTITION_ADDR_WIDTH = BYTES_PER_PARTITION.bit_length() # $clog2(BYTES_PER_PARTITION+1)
PARTITION_ADDR_WIDTH = max(12, PARTITION_ADDR_WIDTH)    # xbars have a minimum m_addr_width of 12 bits
PARTITION_ALIGN = 1 << PARTITION_ADDR_WIDTH
PARTITION_BASE = [BRAM_BASE + i*PARTITION_ALIGN for i in range(NUM_PARTITIONS)]
    
if PROFILE: NUM_DMAS = AVAILABLE_CHANNELS

In [155]:
assert ELEMENT_WIDTH in [16, 32, 64]
assert WORD_WIDTH % ELEMENT_WIDTH == 0
assert ELEMENTS_PER_ROW % ACTIVE_CHANNELS == 0
assert ELEMENTS_PER_ROW % ELEMENTS_PER_WORD == 0
assert NUM_ROWS % ACTIVE_CHANNELS == 0

In [156]:
P = "" if PROFILE else "_0"
BITSTREAM = os.path.join(BASE_DIR, f"hw/fp{ELEMENT_WIDTH}_{MATRIX_DIM}x{MATRIX_DIM}_{int(AXIS_CLK_MHZ)}{P}/design_1.bit")

if DATA_DIR == "mat-gen": MTRX_FILE = os.path.join(BASE_DIR, f"sw/{DATA_DIR}/matrix{ELEMENT_WIDTH}_{MATRIX_DIM}.{FILE_TYPE}")
else: MTRX_FILE = os.path.join(BASE_DIR, f"sw/model/{DATA_DIR}/trmult_{SCALE}{ELEMENT_WIDTH}_padded.{FILE_TYPE}")

### Helpers

In [157]:
#https://github.com/iamNCJ/PYNQ-CDMA-Driver/blob/main/pynq_cdma/cdma.py

from pynq import DefaultIP
from pynq.buffer import PynqBuffer

class CDMA(DefaultIP):
    def __init__(self, description):
        super().__init__(description=description)

    bindto = ['xilinx.com:ip:axi_cdma:4.1']

    @property
    def idle(self):
        """
        `transfer` can only be called when the DMA is idle
        :return: True if the DMA engine is idle
        """
        return self.register_map.CDMASR.Idle

    def _do_transfer(self, source: int, dest: int, bytes_count: int):
        """
        Execute DMA transfer
        :param source: source address
        :param dest: destination address
        :param bytes_count: bytes to transfer
        :return: None
        """            
        if not self.idle:
            raise Exception('DMA transfer can only start when engine is idle')
            
        assert source > 0
        if source > 0xFFFF_FFFF:
            self.register_map.SA_MSB[:] = source >> 8
        self.register_map.SA[:] = source & 0xFFFF_FFFF
                
        assert dest > 0
        if dest > 0xFFFF_FFFF:
            self.register_map.DA_MSB[:] = dest >> 8
        self.register_map.DA[:] = dest & 0xFFFF_FFFF
        
        assert bytes_count > 0
        self.register_map.BTT = bytes_count
        
        # Spin
        while not self.idle:
            pass
        
    def transfer(self, source, dest, bytes_count=None):
        """
        Transfer data through DMA
        :param source:
        :param dest:
        :param bytes_count:
        :return:
        """        
        # Set source
        bytes_source = None
        if isinstance(source, PynqBuffer):
            source_addr = source.device_address
            bytes_source = source.nbytes
        elif isinstance(source, int):
            source_addr = source
        else:
            raise TypeError(f'Type {type(source)} not supported as source')
        
        bytes_dest = None
        if isinstance(dest, PynqBuffer):
            dest_addr = dest.device_address
            bytes_dest = dest.nbytes
        elif isinstance(dest, int):
            dest_addr = dest
        else:
            raise TypeError(f'Type {type(source)} not supported as dest')

        if bytes_count is None:
            if bytes_source is not None and bytes_dest is not None:
                bytes_count = min(bytes_source, bytes_dest)
            elif bytes_source is not None:
                bytes_count = bytes_source
            elif bytes_dest is not None:
                bytes_count = bytes_dest
            else:
                raise RuntimeError(f'Bytes to transfer is not set and cannot be inferred')

        self._do_transfer(source_addr, dest_addr, bytes_count)

In [158]:
from typing import Tuple, Dict, Any

class AxisDmaProfiler:
    """
    Address map:
    
        +0x00: STATUS [0]=have_result, [1]=running
        
        +0x04: CYCLES    [31:0]   +0x08: CYCLES    [63:32]
        +0x0C: BEATS     [31:0]   +0x10: BEATS     [63:32]
        +0x14: BYTES     [31:0]   +0x18: BYTES     [63:32]
        +0x1C: PKTS      [31:0]
        
        +0x20: READY_LOW [31:0]   +0x24: READY_LOW [63:32]
        +0x28: VALID_LOW [31:0]   +0x2C: VALID_LOW [63:32]
        
        +0x30: GAP_LAST  [31:0]   +0x34: GAP_LAST  [63:32]
        +0x38: GAP_TOTAL [31:0]   +0x3C: GAP_TOTAL [63:32]
        +0x40: GAP_MIN   [31:0]   +0x44: GAP_MIN   [63:32]
        +0x48: GAP_MAX   [31:0]   +0x4C: GAP_MAX   [63:32]
        +0x50: GAP_COUNT [31:0]
        
        +0x54: GAP_STATUS [0]=gap_valid, [1]=armed
        
        +0x58: BUSY_TOTAL[31:0]   +0x5C: BUSY_TOTAL[63:32]
    """
    
    OFF_STATUS  = 0x00
    OFF_CYCLES  = 0x04
    OFF_BEATS   = 0x0C
    OFF_BYTES   = 0x14
    OFF_PKTS    = 0x1C
    
    OFF_RDY_LOW = 0x20
    OFF_VLD_LOW = 0x28
    
    OFF_FIRSTHS = 0x30
    
    OFF_BUSY_TOTAL = 0x38
    
    OFF_GAP_STATUS = 0x40
    OFF_GAP_LAST   = 0x44
    OFF_GAP_TOTAL  = 0x4C
    OFF_GAP_MIN    = 0x54
    OFF_GAP_MAX    = 0x5C
    OFF_GAP_COUNT  = 0x64
    
    OFF_GLOB_SS_STATUS = 0x68
    OFF_GLOB_SS_CYCLES = 0x6C
    OFF_GLOB_SM_STATUS = 0x74
    OFF_GLOB_SM_CYCLES = 0x78
    
    CH_WINDOW = 0x100 # 0x100 bytes per port

    def __init__(self, overlay, ports=4, axis_clk_hz=300e6, ip_name_contains="axis_dma_profiler", base=None, size=0x1000):
        
        self.overlay = overlay
        self.ports = ports
        self.axis_clk_hz = axis_clk_hz

        key = next((k for k in overlay.ip_dict if ip_name_contains in k), None)
        
        if key is None:
            if base is None:
                raise ValueError(
                    f"Could not find IP containing '{ip_name_contains}' in overlay.ip_dict. "
                    f"Pass base=... explicitly or adjust ip_name_contains."
                )
        else: 
            ip = overlay.ip_dict[key]
            base = int(ip["phys_addr"])
            size = int(ip["addr_range"])

        if base is None: raise ValueError("Profiler base address is unknown.")
        if size is None: raise ValueError("Profiler size is unknown.")

        self.base = base
        self.size = size
        self.mmio = MMIO(self.base, self.size)
        
    def _rd32(self, off):
        return self.mmio.read(off)

    def _rd64(self, off, attempts=4):
        for _ in range(attempts):
            hi1 = self._rd32(off + 4)
            lo  = self._rd32(off)
            hi2 = self._rd32(off + 4)
            if hi1 == hi2:
                return (hi1 << 32) | lo
        return -1

    def _port_base(self, port):
        if not (0 <= port < self.ports):
            raise ValueError(f"Port out of range: {port} (ports={self.ports})")
        return port * self.CH_WINDOW

    # ========== Public ==========

    def stats(self, port) -> Dict[str, int]:
        """
        Reads (cycles, beats, bytes, pkts). Prefer calling when have_result=1 and running=0.
        """
        b = self._port_base(port)
        cycles  = self._rd64(b + self.OFF_CYCLES)
        beats   = self._rd64(b + self.OFF_BEATS)
        bytes_  = self._rd64(b + self.OFF_BYTES)
        pkts    = self._rd32(b + self.OFF_PKTS)
        
        rdy_lo  = self._rd64(b + self.OFF_RDY_LOW)
        vld_lo  = self._rd64(b + self.OFF_VLD_LOW)
        
        firsths = self._rd64(b + self.OFF_FIRSTHS)
        
        return {
            "cycles":  cycles, 
            "beats":   beats, 
            "bytes":   bytes_, 
            "pkts":    pkts,
            "rdy_lo":  rdy_lo,
            "vld_lo":  vld_lo,
            "firsths": firsths
        }
    
    def gap_stats(self, port) -> Dict[str, Any]:
        """
        Inter-packet gap metrics.
        """
        b = self._port_base(port)
        gap_last   = self._rd64(b + self.OFF_GAP_LAST)
        gap_total  = self._rd64(b + self.OFF_GAP_TOTAL)
        gap_min    = self._rd64(b + self.OFF_GAP_MIN)
        gap_max    = self._rd64(b + self.OFF_GAP_MAX)
        gap_count  = self._rd32(b + self.OFF_GAP_COUNT)
        busy_total = self._rd64(b + self.OFF_BUSY_TOTAL)
        
        if gap_count == 0: gap_last = gap_total = gap_min = gap_max = None

        return {
            "gap_last":   gap_last,
            "gap_total":  gap_total,
            "gap_min":    gap_min,
            "gap_max":    gap_max,
            "gap_count":  gap_count,
            "busy_total": busy_total
        }
    
    def glob_stats(self, port=0) -> Dict[str, int]:
        """
        Get global timers
        """
        b = self._port_base(port)
        ss_cycles = self._rd64(b + self.OFF_GLOB_SS_CYCLES)
        sm_cycles = self._rd64(b + self.OFF_GLOB_SM_CYCLES)
        
        return {
            "ss_cycles": ss_cycles,
            "sm_cycles": sm_cycles
        }
        

    def throughput(self, port) -> Tuple[float, float, Dict[str, int]]:
        """
        Returns (gbps, handshake_utilization, stats) for a completed frame .
        - gbps  = (bytes * 8 / cycles) * axis_clk_hz / 1e9
        - handshake_utilization = beats / cycles   (fraction of cycles that handshook)
        """
        s = self.stats(port)
        cycles = max(1, s["cycles"])
        firsths = s["firsths"]
        gbps = (s["bytes"] * 8.0 * self.axis_clk_hz / (cycles+firsths)) / 1e9
        util = s["beats"] / cycles
        return gbps, util, s
    
    def gap_summary(self, port):
        """
        Analogous to `throughput`: return (g, mean_gap_ns, util_long_run).
          - g: dict with gap aggregates (gap_last, gap_min/max, gap_total, gap_count, busy_total)
          - mean_gap_ns: average inter-packet gap in nanoseconds
          - util_long_run: busy_total / (busy_total + gap_total)
        """
        g = self.gap_stats(port)
        gap_total  = g["gap_total"]
        gap_count  = g["gap_count"]
        busy_total = g["busy_total"]

        mean_gap_cycles = (gap_total / gap_count) if gap_count else 0.0
        mean_gap_ns     = (1e9 * mean_gap_cycles / self.axis_clk_hz) if self.axis_clk_hz else 0.0
        util_long_run   = (busy_total / (busy_total + gap_total)) if gap_total else 0.0

        return g, int(mean_gap_cycles), int(mean_gap_ns), util_long_run

    def __repr__(self) -> str:
        return f"AxisDmaProfiler(base=0x{self.base:08x}, size=0x{self.size:x}, ports={self.ports})"


# Main

### Initialize

In [159]:
if VERBOSE: 
    print(f"\n================================ Parameters =================================")

    print(f"""
CDMA_BASE = 0x{CDMA_BASE:08x}
BRAM_BASE = 0x{BRAM_BASE:08x}
PROF_BASE = 0x{PROF_BASE:08x}

AXIS_CLK_HZ = {AXIS_CLK_HZ}
AXIS_CLK_HZ = {AXIS_CLK_MHZ}

AVAILABLE_CHANNELS = {AVAILABLE_CHANNELS}
ACTIVE_CHANNELS    = {ACTIVE_CHANNELS}
NUM_PARTITIONS     = {NUM_PARTITIONS}

NUM_ROWS           = {NUM_ROWS}
ROWS_PER_CHANNEL   = {ROWS_PER_CHANNEL}

WORD_WIDTH        = {WORD_WIDTH}
ELEMENT_WIDTH     = {ELEMENT_WIDTH}
ELEMENTS_PER_WORD = {ELEMENTS_PER_WORD}

BYTES_PER_ELEMENT = {BYTES_PER_ELEMENT}
BYTES_PER_WORD    = {BYTES_PER_WORD}

ELEMENTS_PER_ROW = {ELEMENTS_PER_ROW}
WORDS_PER_ROW    = {WORDS_PER_ROW}
BYTES_PER_ROW    = {BYTES_PER_ROW}

ELEMENTS_PER_PARTITION = {ELEMENTS_PER_PARTITION}
BYTES_PER_PARTITION    = {BYTES_PER_PARTITION}
PARTITION_ADDR_WIDTH   = {PARTITION_ADDR_WIDTH}
PARTITION_ALIGN        = {PARTITION_ALIGN}
PARTITION_BASE         = {[hex(p) for p in PARTITION_BASE]}""")

matrix = vector = result = None 
PL.reset()

if VERBOSE: print("\n================================== Setup ====================================\n")

# Make sure matrix exists
if not os.path.exists(MTRX_FILE): raise FileNotFoundError(f"No matrix found at {MTRX_FILE}")

# Load overlay
if VERBOSE: print("Loading Overlay.")
overlay = Overlay(BITSTREAM, download=True)
if not overlay.is_loaded():
    raise RuntimeError("Overlay download failed.")
        
# Allocate buffers
if VERBOSE: print("Allocating buffers.")
np_dtype = {16: np.float16, 32: np.float32, 64: np.float64}[ELEMENT_WIDTH]
result = allocate(shape=(ACTIVE_CHANNELS, ROWS_PER_CHANNEL), dtype=np_dtype)
#vector = allocate(shape=(NUM_PARTITIONS, ELEMENTS_PER_PARTITION,), dtype=np_dtype)
vector = allocate(shape=(ELEMENTS_PER_ROW,), dtype=np_dtype)

TILE_MODE = False
tile_rows = ROWS_PER_CHANNEL
while tile_rows > 0:
    try:
        matrix = allocate(shape=(ACTIVE_CHANNELS, tile_rows, ELEMENTS_PER_ROW), dtype=np_dtype)
        if VERBOSE: print(f"    Matrix allocation successful with tile_rows={tile_rows}")
        break
    except RuntimeError:
        if VERBOSE: print(f"    Matrix allocation failed with tile_rows={tile_rows}. ")
        TILE_MODE = True
        tile_rows //= 2
if matrix is None: raise RuntimeError("Could not allocate any CMA buffer for matrix tiles. ")
    
if VERBOSE > 2:
    print(f"    result: addr=0x{result.physical_address:08x}, nbytes=0x{result.nbytes:08x}")
    print(f"    vector: addr=0x{vector.physical_address:08x}, nbytes=0x{vector.nbytes:08x}")
    print(f"    matrix: addr=0x{matrix.physical_address:08x}, nbytes=0x{matrix.nbytes:08x}")

if VERBOSE: print("\n================================ Initialize =================================\n")
    
# Initialize buffers
if VERBOSE: print("Initializing Buffers.")

result.fill(0.0)
#for v in vector: v[:] = (np.random.rand(ELEMENTS_PER_PARTITION) / 100).astype(np_dtype)
vector[:] = (np.random.rand(ELEMENTS_PER_ROW) / 100.0).astype(np_dtype)

if VERBOSE > 1: print(f"    Loading matrix from {MTRX_FILE}")
if   FILE_TYPE == "bin": matrix_data = np.memmap(MTRX_FILE, dtype=np_dtype, mode='r')
elif FILE_TYPE == "npy": matrix_data = np.load(MTRX_FILE, mmap_mode='r')

if matrix_data.shape == (NUM_ROWS, ELEMENTS_PER_ROW):
    matrix_data = matrix_data.reshape((ACTIVE_CHANNELS, ROWS_PER_CHANNEL, ELEMENTS_PER_ROW)) # one matrix per channel
elif matrix_data.shape != (ACTIVE_CHANNELS,ROWS_PER_CHANNEL, ELEMENTS_PER_ROW):
    raise RuntimeError(f"Matrix shape {matrix_data.shape} does not match expected {(NUM_ROWS, ELEMENTS_PER_ROW)}.")

if not TILE_MODE: np.copyto(matrix, matrix_data)

result.flush()
vector.flush()
matrix.flush()

# Initialize profiler
if PROFILE: 
    if VERBOSE: print("Initializing Profiler.")
    prof = AxisDmaProfiler(overlay, ports=NUM_DMAS, axis_clk_hz=AXIS_CLK_HZ, base=PROF_BASE)
    if VERBOSE > 2: print(f"    {prof.__repr__()}")

# Initialize DMAs
if VERBOSE: print("Initializing DMAs.")
dmas = [
    getattr(overlay, name)
    for name in sorted(overlay.ip_dict, key=lambda n: int(re.search(r"\d+$", n).group()))
    if name.startswith("axi_dma")
]
if VERBOSE > 2: print("    dmas =", [d.description['fullpath'] for d in dmas])
if len(dmas) != AVAILABLE_CHANNELS:
    raise RuntimeError(f"Check overlay. Found {len(dmas)} DMAs with AVAILABLE_CHANNELS={AVAILABLE_CHANNELS}.")
    
# Initialize CDMA
if VERBOSE: print("Initializing CDMA.")
cdma = CDMA(overlay.ip_dict['axi_cdma_0'])
cdma_addr = hex(overlay.ip_dict['axi_cdma_0']['phys_addr'])
print(f"    cdma = axi_cdma_0@{cdma_addr}")
        
if VERBOSE: print("\n=============================================================================\n")


================================ Parameters =================================

CDMA_BASE = 0xa0000000
BRAM_BASE = 0x80000000
PROF_BASE = 0xa00c0000

AXIS_CLK_HZ = 250000000.0
AXIS_CLK_HZ = 250.0

AVAILABLE_CHANNELS = 4
ACTIVE_CHANNELS    = 4
NUM_PARTITIONS     = 4

NUM_ROWS           = 512
ROWS_PER_CHANNEL   = 128

WORD_WIDTH        = 128
ELEMENT_WIDTH     = 16
ELEMENTS_PER_WORD = 8

BYTES_PER_ELEMENT = 2
BYTES_PER_WORD    = 16

ELEMENTS_PER_ROW = 512
WORDS_PER_ROW    = 64
BYTES_PER_ROW    = 1024

ELEMENTS_PER_PARTITION = 128
BYTES_PER_PARTITION    = 256
PARTITION_ADDR_WIDTH   = 12
PARTITION_ALIGN        = 4096
PARTITION_BASE         = ['0x80000000', '0x80001000', '0x80002000', '0x80003000']

================================== Setup ====================================

Loading Overlay.
Allocating buffers.
    Matrix allocation successful with tile_rows=128
    result: addr=0x20a43000, nbytes=0x00000400
    vector: addr=0x20a42000, nbytes=0x00000400
    matrix: addr=0x37c80000, nbytes

### Kernel Functions

In [160]:
def send_vec(n):
        
    print(f"\n============================== Send Vector ({n}) ==============================\n")

    # Write vector to BRAM
    if VERBOSE: print("Writing vector to BRAM.")
    
    start = time.perf_counter()
    
    #for p in range(NUM_PARTITIONS):
    #    start_idx = p*ELEMENTS_PER_PARTITION
    #    if VERBOSE > 2: 
    #        print(f"    cdma@{cdma_addr}.transfer("
    #              f"src=0x{vector[p].physical_address:08x}, dest=0x{PARTITION_BASE[p]:08x})")
    #    cdma.transfer(source=vector[p], dest=PARTITION_BASE[p], bytes_count=BYTES_PER_PARTITION)
    print(f"    cdma@{cdma_addr}.transfer(src=0x{vector.physical_address:08x}, dest=0x{PARTITION_BASE[0]:08x})")
    cdma.transfer(source=vector, dest=PARTITION_BASE[0], bytes_count=BYTES_PER_ROW)
    
    return (time.perf_counter() - start) * 1000

def comp_full():
    start = time.perf_counter()
    for ch, d in enumerate(dmas): d.recvchannel.transfer(result[ch])
    for ch, d in enumerate(dmas): d.sendchannel.transfer(matrix[ch])
    for d in dmas: d.sendchannel.wait()
    for d in dmas: d.recvchannel.wait()
    return (time.perf_counter() - start) * 1000

def comp_tiled():
    start = time.perf_counter()
    for ch, d in enumerate(dmas): d.recvchannel.transfer(result[ch])

    for tile_start in range(0, ROWS_PER_CHANNEL, tile_rows):
        
        tile_end = min(tile_start + tile_rows, ROWS_PER_CHANNEL)
        rows_this_tile = tile_end - tile_start
        if VERBOSE > 1: print(f"Tile rows [{tile_start}:{tile_end-1}] -> rows_this_tile={rows_this_tile}")
        
        np.copyto(matrix[:, :rows_this_tile, :], matrix_data[:, tile_start:tile_end, :])
        matrix.flush()
        
        for ch, d in enumerate(dmas): d.sendchannel.transfer(matrix[ch, :rows_this_tile, :])
        for ch, d in enumerate(dmas): d.sendchannel.wait()

    for d in dmas: d.recvchannel.wait()
    return (time.perf_counter() - start) * 1000

def run_once(n):

    if VERBOSE > 1: print(f"\n================================ Compute ({n}) ================================\n")

    if not TILE_MODE: comp_time_ms = comp_full()
    else:             comp_time_ms = comp_tiled()

    # Print results
    errs = np.array([])
    if VERBOSE > 1:
        if VERBOSE > 2: print()
        for i in range(ACTIVE_CHANNELS):
            for j in range(ROWS_PER_CHANNEL):
                row_idx = i * ROWS_PER_CHANNEL + j
                if (row_idx > 5 and row_idx < NUM_ROWS-5):
                    if not VERBOSE > 3:
                        if (row_idx == 6): print("...")
                        continue
                actual = result[i][j].astype(np.float64)
                expected = float(np.dot(matrix_data[i][j].astype(np.float64), vector.flatten().astype(np.float64)))
                err = abs(actual - expected) / expected
                if expected != 0: errs = np.append(errs, err)

                print(f"Row {row_idx}: Actual={actual:.32f} | Expected={expected:.32f} | Error={abs_err:.6f}")

        print(f"Average error: {np.mean(err):.3g} (result_avg={np.mean(result.astype(np.float64).flatten()):.3g})")

    print(f"\n============================== Performance ({n}) ==============================\n")

    if PROFILE:
        for i in range(prof.ports):
            gbps, util, s = prof.throughput(port=i)
            g, mean_gap_cycles, mean_gap_ns, util_long_run = prof.gap_summary(port=i)
            print(f"Port {i}:")
            print(f"    Handshake util: {util:.3%} | Throughput: {gbps/8:.3f} GB/s")
            if g["gap_count"] > 0: print(f"    Avg gap len: {mean_gap_cycles} cycles, {mean_gap_ns} ns | Busy util: {util_long_run:.2%}")
            if VERBOSE:
                print(f"    {s}")
                if g["gap_count"] > 0: print(f"    {g}")

        glob = prof.glob_stats()
        ss_cycles = glob['ss_cycles']
        sm_cycles = glob['sm_cycles']
        
        ss_time_ms = ss_cycles * (1 / AXIS_CLK_HZ) * 1e3
        sm_time_ms = sm_cycles * (1 / AXIS_CLK_HZ) * 1e3
        throughput_GBs = (NUM_ROWS * ELEMENTS_PER_ROW * BYTES_PER_ELEMENT / 1e9) / (sm_time_ms / 1e3) if sm_time_ms != 0 else 0
        
        print(f"\nTotal throughput: {throughput_GBs:.3f} GB/s")
        print(f"RTL SS time: {ss_cycles} cycles -> {ss_time_ms:.3f} ms")
        print(f"RTL SM time: {sm_cycles} cycles -> {sm_time_ms:.3f} ms")
        
    result.invalidate()
    return comp_time_ms

### Execute

In [161]:
NUM_RUNS = 4

overlay.download()

for n in range(NUM_RUNS):
    vec_time_ms = send_vec()
    print(f"Elapsed time: {vec_time_ms:.6f} ms")
    
    comp_time_ms = run_once(n)
    print(f"Python compute time: {comp_time_ms:.3f} ms")

print("\n=============================================================================\n")


============================== Send Vector ==============================

Writing vector to BRAM.
    cdma@0xa0000000.transfer(src=0x20a42000, dest=0x80000000)
Elapsed time: 37.060867 ms

================================ Compute (0) ================================


Row 0: Actual=123.93750000000000000000000000000000 | Expected=123.98887855513021349906921386718750 | Error=0.051379
Row 1: Actual=122.25000000000000000000000000000000 | Expected=122.25662419246509671211242675781250 | Error=0.006624
Row 2: Actual=129.75000000000000000000000000000000 | Expected=129.77316379779949784278869628906250 | Error=0.023164
Row 3: Actual=122.12500000000000000000000000000000 | Expected=122.15515493950806558132171630859375 | Error=0.030155
Row 4: Actual=120.62500000000000000000000000000000 | Expected=120.68925086734816431999206542968750 | Error=0.064251
Row 5: Actual=125.87500000000000000000000000000000 | Expected=125.93240787804825231432914733886719 | Error=0.057408
...
Row 507: Actual=130.0000000000

### Cleanup

In [162]:
for buf in [matrix, vector, result]: 
    if buf is not None: buf.freebuffer()